# GRPO on the neurarch verifiable-reward environment (full run)

**Before you start:** Runtime -> Change runtime type -> **T4 GPU** (not A100/L4; they burn compute units 6-8x faster).

PRIVATE repo: upload `neurarch-arch-bench.zip` via the Files panel (left) first. PUBLIC repo: cell 1 clones it.

Run cells top to bottom. Total ~1.5-2h, ~3 compute units. **Disconnect the runtime when done** (idle paid runtimes keep charging).


In [ ]:
# 1. Get the repo, install deps, start the reward server.
import os
if not os.path.isdir('neurarch-arch-bench'):
    if os.path.exists('neurarch-arch-bench.zip'):
        !unzip -q neurarch-arch-bench.zip
    else:
        !git clone --depth 1 https://github.com/neurarch-ai/neurarch-arch-bench
%cd neurarch-arch-bench
!pip -q install 'trl>=0.14' transformers datasets accelerate peft bitsandbytes
!pip -q uninstall -y torchao   # Colab's old torchao breaks PEFT's LoRA path
import subprocess, time, urllib.request, json
subprocess.Popen(['node','env-server.mjs']); time.sleep(3)
print('reward server:', json.load(urllib.request.urlopen('http://localhost:8737/health')))
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime to T4!')


## 2. Baseline (untrained) pass@1 on the held-out seed-999 split


In [ ]:
!python training/train_grpo.py --eval-only --seed 999 --count 64


## 3. Train (GRPO, LoRA, ~1.5h on T4)
lr 1e-5 and short completions are what make a small model actually learn here.
Watch the first ~40 steps: `rewards/arch_reward/mean` trending up = good;
stuck at -0.5 with `clipped_ratio`->1 = lr too high, stop and rerun with `--lr 5e-6`.


In [ ]:
!python training/train_grpo.py --steps 250 --count 256 --seed 123 --lora --lr 1e-5 --max-completion 384


## 4. Eval every checkpoint + print one summary table
Paste the printed table back to get the learning curve + paper update.


In [ ]:
import subprocess, re, glob, os
def ev(model):
    out = subprocess.run(['python','training/train_grpo.py','--eval-only','--seed','999','--count','64','--model',model],
                         capture_output=True, text=True).stdout
    p=re.search(r'pass@1: (\d+)/(\d+)', out); pf=re.search(r'parse failures: (\d+)', out); mr=re.search(r'mean reward: ([\d.]+)', out)
    return (f'{p.group(1)}/{p.group(2)}' if p else '?', pf.group(1) if pf else '?', mr.group(1) if mr else '?')
def stepkey(path):
    b=os.path.basename(path)
    if b.endswith('final'): return 10**9
    m=re.search(r'checkpoint-(\d+)', b); return int(m.group(1)) if m else 0
rows=[('baseline','Qwen/Qwen2.5-1.5B-Instruct')]
rows += [(os.path.basename(c), c) for c in sorted(glob.glob('out/grpo-arch/checkpoint-*'), key=stepkey)]
print('| checkpoint | pass@1 | parse fails | mean reward |'); print('|---|---|---|---|')
for name, model in rows:
    p, pf, mr = ev(model); print(f'| {name} | {p} | {pf}/64 | {mr} |')


**Done.** Copy the table above and paste it back. Then Runtime -> Disconnect and delete runtime to stop billing.
